# Compare pywatershed and PRMS

This notebook compares pywatershed (PWS) and PRMS outputs. It is intended to give users insight on and tools to examine the similarity of pywatershed and PRMS simulations. While these are very close in most cases, this notebook makes it easy to find when they are not close by providing statistics at individual HRUs and timeseries plots for variables on all HRUs or stream segments.

The PRMS side of the comparison is the pws-prms-oracle answer set: PRMS 5.2.1 output for each test domain (`hru_1`, `drb_2yr`, `ucb_2yr`, ...), populated into `test_data/<domain>/output/` by `test_data/generate/oracle_populate.py` (see DEVELOPER.md, Testing; `autotest/ci_local.sh` does it for you). An editable install of pywatershed (`pip install -e` in the repository root) is still needed to locate `test_data/`. Any other domain may be used; see below for notes on setting up your own.

## Precision of PRMS
The runs of PRMS in this notebook use PRMS binaries where all floating point variables are double precision (64 bit). If you are running PRMS yourself, you are likely using a version
with a mix of single (32 bit) and double precision floating point variables. There are some differences between PRMS built in these two ways. If you want to adapt this notebook to compare against mixed precision PRMS, please contact the pywatershed developers. Because we do not want to implement mixed precision operations in pywatershed, its output is compared to PRMS built with only double precision floating point variables. The oracle builds PRMS 5.2.1 with the `DBL_PREC=true` flag; its source lives in the pws-prms-oracle repository.

## Notes on setting up other domains
If want to supply your own domain and see how pywatershed works on it, here are notes on doing so. Domains must supply the correct, required files in `test_data/your_domain` which are given in this listing:

```
nhm.control  prcp.cbh      sf_data       tmax.nc       tmin.nc
myparam.param prcp.nc       tmax.cbh      tmin.cbh
```

The `*.cbh` files must be pre-converted to netcdf for `prcp`, `tmin`, and `tmax` and how to do this can be found near the top of notebook 02. `output/` must hold that domain's PRMS output converted to netCDF, one variable per file, as the oracle produces it. The `nhm.control` and `myparam.param` files are used by both PRMS and PWS. The `nhm.control` files in the repo are specific for being able to run sub-models and include a nearly maximal amount of model output (time-inefficient for both PRMS and PWS). The stock control files can be found in `test_data/common` there is a file for single-hru domains and multi-hru domains and these are identical (as appropriate) for the domains included in the repository. For running a large domain, for example, it is desirable to reduce the total amount of output (but this may not allow for PWS sub-models to be run as PRMS dosent necessarily supply all the required fields). So you may modify the `nhm.control` file but take careful note of what options are available in pywatershed as currently only NHM configuration is available.

## Plots
All of the code required for plotting below is included so that it can be further tailored to your tastes.


## Imports, etc

In [ ]:
import pathlib as pl
from shutil import rmtree
import warnings

import hvplot.pandas  # noqa
import hvplot.xarray  # noqa
import jupyter_black
import numpy as np
import pywatershed as pws
import xarray as xr

nb_output_dir = pl.Path("./03_compare_pws_prms")

jupyter_black.load()  # auto-format the code in this notebook

## Configuration

Specify what you want!

In [ ]:
domain_name: str = "drb_2yr"  # must be present in test_data/domain_name
calc_method: str = "numba"
imbalance_behavior: str = None

run_pws: bool = True  # run if the output does not exist on disk
force_pws_run: bool = True  # if it exists on disk, re-run it and overwrite?

## PRMS output

In [ ]:
domain_dir = pws.utils.get_test_data_dir() / domain_name

PRMS is not run here; `domain_dir / "output"` is the oracle's answer set for this domain.

In [ ]:
prms_output_dir = domain_dir / "output"
assert prms_output_dir.exists(), (
    f"{prms_output_dir} missing: populate the domain from the oracle "
    "(DEVELOPER.md, Testing)"
)

## Run pywatershed

In [ ]:
if run_pws:
    nhm_processes = [
        pws.PRMSSolarGeometry,  # submodles are possible
        pws.PRMSAtmosphere,
        pws.PRMSCanopy,
        pws.PRMSSnow,
        pws.PRMSRunoff,
        pws.PRMSSoilzone,
        pws.PRMSGroundwater,
        pws.PRMSChannel,
    ]

    if len(nhm_processes) == 8:
        input_dir = domain_dir
        run_dir = nb_output_dir / f"{domain_name}_full_nhm"
    else:
        input_dir = domain_dir / "output"
        run_dir = nb_output_dir / f"{domain_name}_subset_nhm"

    control = pws.Control.load_prms(
        domain_dir / "nhm.control", warn_unused_options=False
    )
    output_var_names = control.options["netcdf_output_var_names"] + [
        "soltab_horad_potsw",
        "soltab_potsw",
        "soltab_sunhrs",
    ]
    control.options = control.options | {
        "input_dir": input_dir,
        "imbalance_behavior": imbalance_behavior,
        "calc_method": calc_method,
        "netcdf_output_dir": run_dir,
        "netcdf_output_var_names": output_var_names,
    }

    params = pws.parameters.PrmsParameters.load(domain_dir / "myparam.param")

    if run_dir.exists():
        if force_pws_run:
            rmtree(run_dir)
        else:
            raise RuntimeError("run directory exists")

    print(f"PWS writing output to {run_dir}")

    nhm = pws.Model(
        nhm_processes,
        control=control,
        parameters=params,
    )
    nhm.run(finalize=True)

## Compare outputs

In [ ]:
def compare_var_timeseries(var_name, rmse_min=None):
    """Plots compare timeseries a PWS and PRMS variable for all locations in domain (scrollable).

    Args:
        var_name: string name of variable
        rmse_min: only plot locations which exceed this minimum rmse between PRMS and PWS.

    """
    from textwrap import fill

    var_meta = pws.meta.find_variables(var_name)[var_name]
    ylabel = f"{fill(var_meta['desc'], 40)}\n({var_meta['units']})"

    pws_file = run_dir / f"{var_name}.nc"
    prms_file = domain_dir / f"output/{var_name}.nc"
    if not prms_file.exists():
        return None
    if not pws_file.exists():
        print(f"PWS file {pws_file} DNE, skipping.")
        return None

    prms_var = xr.open_dataarray(prms_file)
    pws_var = xr.open_dataarray(pws_file)

    if rmse_min is not None:
        if "time" in prms_var.dims:
            time_dim = "time"
        else:
            time_dim = "doy"

        rmse = np.sqrt((pws_var - prms_var).mean(dim=time_dim) ** 2)
        mask_ge_min = rmse >= rmse_min
        n_mask = len(np.where(mask_ge_min)[0])
        print(f"There are {n_mask} locations with RMSE > {rmse_min}")
        if n_mask == 0:
            return None
        prms_var = prms_var.where(mask_ge_min, drop=True)
        pws_var = pws_var.where(mask_ge_min, drop=True)

    comp_ds = xr.merge(
        [
            prms_var.rename("prms"),
            pws_var.rename("pws"),
        ]
    )
    var_meta = pws.meta.find_variables(var_name)[var_name]
    space_coord = list(comp_ds.coords)
    for t_coord in ["doy", "time"]:
        if t_coord in space_coord:
            space_coord.remove(t_coord)

    display(
        comp_ds.hvplot(
            frame_width=800,
            frame_height=500,
            groupby=space_coord,
            # title=title,
            ylabel=ylabel,
            group_label="Model",
        )
    )

In [ ]:
def calc_stat_location(var_name, stat_name):
    """Calculate a statistic location-wise (over time).

    Args:
        var_name: str for the variable of interest
        stat_name: one of ["rmse", "rrmse"]
    """
    prms_file = domain_dir / f"output/{var_name}.nc"
    if not prms_file.exists():
        print(f"PRMS file '{prms_file}' DNE, skipping.")
        return None
    prms = xr.open_dataarray(prms_file, decode_timedelta=False)
    pws_file = run_dir / f"{var_name}.nc"
    if not pws_file.exists():
        print(f"PWS file '{prms_file}' DNE, skipping.")
        return None
    nhm_after = xr.open_dataarray(pws_file, decode_timedelta=False)
    if "time" in prms.dims:
        time_dim = "time"
    else:
        time_dim = "doy"
    if stat_name.lower() == "rmse":
        stat = np.sqrt((nhm_after - prms).mean(dim=time_dim) ** 2)
    elif stat_name.lower() == "rrmse":
        stat = np.sqrt((nhm_after - prms).mean(dim=time_dim) ** 2)
        stat = stat / prms.mean(dim=time_dim)  # rrmse
    return stat.to_dataframe().melt(ignore_index=False)


def box_jitter_plot(
    df, subplot_width: int = 400, stat_name: str = "Statistic"
):
    """Box/violin-plot of a dataframe.

    Args:
        df: a pd.Dataframe
        subplot_width: int for how wide the subplots should be
        stat_name: str of the statisitc name
    """
    from textwrap import fill

    var_name = df.variable.iloc[0]
    var_meta = pws.meta.find_variables(var_name)[var_name]
    ylabel = (
        f"{stat_name} of\n{fill(var_meta['desc'], 40)}\n({var_meta['units']})"
    )
    coord = df.index.name

    box = df.hvplot.violin(y="value", by="variable", legend=False)
    jitter = df.hvplot.scatter(
        y="value",
        x="variable",
        hover_cols=[coord],
    )
    return (box * jitter).opts(
        width=subplot_width,
        # xlabel=f"over {coord}s",
        xlabel="",
        ylabel=ylabel,
    )


def plot_proc_stats(
    proc, stat_name: str = "RMSE", ncols: int = 5, subplot_width: int = 300
):
    """Plot pywatershed process stats.

    For a process (e.g. pws.PRMSRunoff), make box/violin plots of its stats for each of its (available) variables

    Args:
        proc: a pws.Process subclass
        stat_name: string of the statistic desired to be passed to box_jitter_plot
        ncols: int number of columns in the plot
        subplot_widt: int width of the subplots

    """
    var_stats = []
    for var_name in proc.get_variables():
        var_stats += [calc_stat_location(var_name, stat_name)]

    var_plots = [
        box_jitter_plot(vv, subplot_width=subplot_width, stat_name=stat_name)
        for vv in var_stats
        if vv is not None
    ]
    if len(var_plots) == 0:
        return None

    plot = var_plots[0]
    for vv in var_plots[1:]:
        plot += vv

    plot = plot.opts(shared_axes=False)
    if len(var_plots) > 1:
        plot = plot.cols(ncols)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        display(plot)

In [ ]:
if pws.PRMSSolarGeometry in nhm_processes:
    plot_proc_stats(pws.PRMSSolarGeometry, "RMSE", 5)

In [ ]:
if pws.PRMSSolarGeometry in nhm_processes:
    compare_var_timeseries("soltab_potsw")

In [ ]:
if pws.PRMSAtmosphere in nhm_processes:
    plot_proc_stats(pws.PRMSAtmosphere, "RMSE", 4)

In [ ]:
if pws.PRMSAtmosphere in nhm_processes:
    compare_var_timeseries("tmaxf")

In [ ]:
if pws.PRMSCanopy in nhm_processes:
    plot_proc_stats(pws.PRMSCanopy, "RMSE", 4)

In [ ]:
if pws.PRMSCanopy in nhm_processes:
    compare_var_timeseries("intcp_stor")

In [ ]:
if pws.PRMSSnow in nhm_processes:
    plot_proc_stats(pws.PRMSSnow, "RMSE", 4)

In [ ]:
if pws.PRMSSnow in nhm_processes:
    compare_var_timeseries("pkwater_equiv")

In [ ]:
if pws.PRMSRunoff in nhm_processes:
    plot_proc_stats(pws.PRMSRunoff, "RMSE", 4)

In [ ]:
if pws.PRMSRunoff in nhm_processes:
    compare_var_timeseries("contrib_fraction")

In [ ]:
if pws.PRMSSoilzone in nhm_processes:
    plot_proc_stats(pws.PRMSSoilzone, "RMSE", 4)

In [ ]:
if pws.PRMSSoilzone in nhm_processes:
    compare_var_timeseries("soil_rechr")

In [ ]:
if pws.PRMSGroundwater in nhm_processes:
    plot_proc_stats(pws.PRMSGroundwater, "RMSE", 4)

In [ ]:
if pws.PRMSGroundwater in nhm_processes:
    compare_var_timeseries("gwres_flow")

In [ ]:
if pws.PRMSChannel in nhm_processes:
    plot_proc_stats(pws.PRMSChannel, "RMSE", 4)

In [ ]:
if pws.PRMSChannel in nhm_processes:
    compare_var_timeseries("seg_outflow")  # , rmse_min=0.01)